### PyQASM Branch optimization 
Adapted from [qBraid docs](https://docs.qbraid.com/pyqasm/user-guide/advanced-features).

#### Branch elimination
We evaluate the outcome of the branch condition, if it is known at compile time and does not contain measurement results of qubits. We remove the branch statement and if it evaluates to a true value, attach the corresponding block of code to the main program.

In [1]:
import pyqasm

qasm_code = """
OPENQASM 3.0;
include "stdgates.inc";
qubit[1] q;
bit[1] c;
int[32] a = 0;
if(a > 0){
h q[0];
}
if(a < 0){
x q[0];
}
if(a == 0){
y q[0];
measure q -> c;
}
"""
module = pyqasm.loads(qasm_code)
module.unroll()

print(pyqasm.dumps(module))

OPENQASM 3.0;
include "stdgates.inc";
qubit[1] q;
bit[1] c;
y q[0];
c[0] = measure q[0];



#### Branch unrolling
If the branch condition contains measurement results of qubits, we unfold the classical registers into individual bits and insert equivalent conditional statements for each bit. This method is particularly useful for systems which do not support multi-bit classical registers in conditional statements.

In [2]:
qasm_code = """OPENQASM 3.0;
include "stdgates.inc";
qubit[1] q;
bit[4] c;
if(c == 3){
h q[0];
}
if(c >= 3){
h q[0];
} else {
x q[0];
}
if(c <= 3){
h q[0];
} else {
x q[0];
}
if(c < 4){
h q[0];
} else {
x q[0];
}
"""
module = pyqasm.loads(qasm_code)
module.unroll()

print(pyqasm.dumps(module))

OPENQASM 3.0;
include "stdgates.inc";
qubit[1] q;
bit[4] c;
if (c[0] == false) {
  if (c[1] == false) {
    if (c[2] == true) {
      if (c[3] == true) {
        h q[0];
      }
    }
  }
}
if (c[2] == true) {
  if (c[3] == true) {
    h q[0];
  } else {
    x q[0];
  }
} else {
  x q[0];
}
if (c[0] == false) {
  if (c[1] == false) {
    h q[0];
  } else {
    x q[0];
  }
} else {
  x q[0];
}
if (c[0] == false) {
  if (c[1] == false) {
    h q[0];
  } else {
    x q[0];
  }
} else {
  x q[0];
}



### Switch Case Optimization

OpenQASM 3 enforces the switch targets to be of type int and the cases to be unique integer literals or constant integer expressions. This implies that the switch case statements can be optimized at compile time _as long as the target variable is not dependent on a measurement result_. **This is the capability we wish to expand in UCC**

Once the target variable is evaluated at compile time, the switch statement is removed and the corresponding switch case code block is attached to the main program.

In [3]:
qasm_code = """OPENQASM 3.0;
include "stdgates.inc";
qubit[1] q;
bit[4] c;
if(c == 3){
h q[0];
}
if(c >= 3){
h q[0];
} else {
x q[0];
}
if(c <= 3){
h q[0];
} else {
x q[0];
}
if(c < 4){
h q[0];
} else {
x q[0];
}
"""
module = pyqasm.loads(qasm_code)
module.unroll()

print(pyqasm.dumps(module))

OPENQASM 3.0;
include "stdgates.inc";
qubit[1] q;
bit[4] c;
if (c[0] == false) {
  if (c[1] == false) {
    if (c[2] == true) {
      if (c[3] == true) {
        h q[0];
      }
    }
  }
}
if (c[2] == true) {
  if (c[3] == true) {
    h q[0];
  } else {
    x q[0];
  }
} else {
  x q[0];
}
if (c[0] == false) {
  if (c[1] == false) {
    h q[0];
  } else {
    x q[0];
  }
} else {
  x q[0];
}
if (c[0] == false) {
  if (c[1] == false) {
    h q[0];
  } else {
    x q[0];
  }
} else {
  x q[0];
}



#### IF-ELSE conditioned on a qubit measurement result

In [4]:
qasm_code = """OPENQASM 3.0;
include "stdgates.inc";
qubit[2] q;
bit[1] c;  // Use a 1-bit register for measurement

// Measure qubit 1 into c[0]
c[0] = measure q[1];

// Apply operations based on c[0]
if (c[0] == 0) {
  h q[0];
} else {
  x q[0];
}"""

module = pyqasm.loads(qasm_code)
module.unroll()

print(pyqasm.dumps(module))

OPENQASM 3.0;
include "stdgates.inc";
qubit[2] q;
bit[1] c;
c[0] = measure q[1];
if (c[0] == false) {
  h q[0];
} else {
  x q[0];
}



### FOR loop conditioned on mesasurement result
QASM 3.0 does not allow you to specify an indeterminate length loop. You have to specify a set number of iterations. 

For instance if we wanted to code a small repetition code:

In [17]:
qasm_code = """OPENQASM 3.0;
include "stdgates.inc";

qubit[5] q;
bit[2] syndrome;
const uint max_iterations = 3;  // Valid constant declaration

// Encoding
reset q[0];
cx q[0], q[1];
cx q[0], q[2];

int i = 0;

// Keep applying hadamards and measuring a qubit
// until 10, |1>s are measured
while (i < 10) {
    reset q[3];
    reset q[4];
    
    // First stabilizer
    cx q[0], q[3];
    cx q[1], q[3];
    measure q[3] -> syndrome[0];
    
    // Second stabilizer
    cx q[1], q[4];
    cx q[2], q[4];
    measure q[4] -> syndrome[1];
    
    // Correction logic
    if (syndrome == 1) {
        x q[2];
    } else if (syndrome == 2) {
        x q[0];
    } else if (syndrome == 3) {
        x q[1];
    }
}

// Final measurement
bit[3] result;
measure q[0] -> result[0];
measure q[1] -> result[1];
measure q[2] -> result[2];"""


module = pyqasm.loads(qasm_code)
# module.unroll()

print(pyqasm.dumps(module))

OPENQASM 3.0;
include "stdgates.inc";
qubit[5] q;
bit[2] syndrome;
const uint max_iterations = 3;
reset q[0];
cx q[0], q[1];
cx q[0], q[2];
int i = 0;
while (i < 10) {
  reset q[3];
  reset q[4];
  cx q[0], q[3];
  cx q[1], q[3];
  syndrome[0] = measure q[3];
  cx q[1], q[4];
  cx q[2], q[4];
  syndrome[1] = measure q[4];
  if (syndrome == 1) {
    x q[2];
  } else if (syndrome == 2) {
    x q[0];
  } else if (syndrome == 3) {
    x q[1];
  }
}
bit[3] result;
result[0] = measure q[0];
result[1] = measure q[1];
result[2] = measure q[2];



In [14]:
qasm_code = """OPENQASM 3.0;
include "stdgates.inc";
qubit q;
bit result;

int i = 0;
// Keep applying hadamards and measuring a qubit
// until 10, |1>s are measured
while (i < 10) {
    h q;
    result = measure q;
    if (result) {
        i += 1;
    }
}"""

module = pyqasm.loads(qasm_code)
module.unroll()

print(pyqasm.dumps(module))

ERROR:root:Error at line 9, column 0 in QASM file


ValidationError: Unsupported statement of type <class 'openqasm3.ast.WhileLoop'>

#### Version with pre-determined number of iterations in FOR loop

In [15]:
qasm_code = """OPENQASM 3.0;
include "stdgates.inc";

qubit[5] q;
bit[2] syndrome;
const uint max_iterations = 3;  // Valid constant declaration

// Encoding
reset q[0];
cx q[0], q[1];
cx q[0], q[2];

// Stabilization loop (verified syntax)
for int i in [0:max_iterations] {  // OFFICIAL SYNTAX: [start:end]
    reset q[3];
    reset q[4];
    
    // First stabilizer
    cx q[0], q[3];
    cx q[1], q[3];
    measure q[3] -> syndrome[0];
    
    // Second stabilizer
    cx q[1], q[4];
    cx q[2], q[4];
    measure q[4] -> syndrome[1];
    
    // Correction logic
    if (syndrome == 1) {
        x q[2];
    } else if (syndrome == 2) {
        x q[0];
    } else if (syndrome == 3) {
        x q[1];
    }
}

// Final measurement
bit[3] result;
measure q[0] -> result[0];
measure q[1] -> result[1];
measure q[2] -> result[2];"""


module = pyqasm.loads(qasm_code)
module.unroll()

print(pyqasm.dumps(module))

OPENQASM 3.0;
include "stdgates.inc";
qubit[5] q;
bit[2] syndrome;
reset q[0];
cx q[0], q[1];
cx q[0], q[2];
reset q[3];
reset q[4];
cx q[0], q[3];
cx q[1], q[3];
syndrome[0] = measure q[3];
cx q[1], q[4];
cx q[2], q[4];
syndrome[1] = measure q[4];
if (syndrome[0] == false) {
  if (syndrome[1] == true) {
    x q[2];
  } else if (syndrome[0] == true) {
    if (syndrome[1] == false) {
      x q[0];
    } else if (syndrome[0] == true) {
      if (syndrome[1] == true) {
        x q[1];
      }
    }
  } else if (syndrome[0] == true) {
    if (syndrome[1] == true) {
      x q[1];
    }
  }
} else if (syndrome[0] == true) {
  if (syndrome[1] == false) {
    x q[0];
  } else if (syndrome[0] == true) {
    if (syndrome[1] == true) {
      x q[1];
    }
  }
} else if (syndrome[0] == true) {
  if (syndrome[1] == true) {
    x q[1];
  }
}
reset q[3];
reset q[4];
cx q[0], q[3];
cx q[1], q[3];
syndrome[0] = measure q[3];
cx q[1], q[4];
cx q[2], q[4];
syndrome[1] = measure q[4];
if (syndrome[0] == f

In [18]:
# Assume a pre-determined number of QEC rounds